# Train the circuit board defect detector (RF-DETR) on Colab

Fine-tunes **RF-DETR Nano** to find four defects on circuit boards (bent pins, melted
solder, scratches, missing components), on the **VisA** dataset (PCB1-3). The weights it
produces run on a CPU in `cv-conveyor`, the simulated conveyor that sends its results to
Odoo's Vision Inspection.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
About an hour on a T4: downloading VisA (1.9 GB) takes a few minutes, training most of it.

Licences: VisA © Amazon, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/);
RF-DETR Nano, Apache-2.0; this project, Apache-2.0.

## 1. GPU

In [ ]:
!nvidia-smi -L

## 2. The project

The same commands as on a laptop: `cv-prepare`, `cv-train`, `cv-evaluate`.

In [ ]:
!git clone -q https://github.com/EricRForgeFlow/cv_quality_edge_demo.git
%cd cv_quality_edge_demo
!pip install -q -e ".[train]"

## 3. The dataset

Downloads VisA, keeps PCB1-3 and turns their defect masks into boxes (COCO format).

In [ ]:
!cv-prepare

## 4. Training

`nano` is the model the conveyor runs on a CPU. If its scores below are poor, set
`MODEL = "small"` (also Apache-2.0) and run this cell and the next ones again.

In [ ]:
MODEL = "nano"
!cv-train --preset colab --model {MODEL}

## 5. Scores on the test boards (never seen in training)

`map50`: how well it finds each defect where it is. `boards`: what quality control cares
about: defective boards caught (`recall`) and how often a rejected board really was
defective (`precision`).

In [ ]:
!cv-evaluate --model models/pcb_{MODEL}.pth

## 6. A look at a few test boards

In [ ]:
import glob, json
from IPython.display import Image, display

with open("dataset/test/_annotations.coco.json") as handle:
    coco = json.load(handle)
defective = {a["image_id"] for a in coco["annotations"]}
samples = [i["file_name"] for i in coco["images"] if i["id"] in defective][:4]
samples += [i["file_name"] for i in coco["images"] if i["id"] not in defective][:2]
paths = " ".join(f"dataset/test/{name}" for name in samples)
!cv-detect {paths} --model models/pcb_{MODEL}.pth
for name in samples:
    display(Image(f"runs/detect/{name}", width=640))

## 7. Keep the weights

Copies the weights and their scores to Google Drive (`MyDrive/cv_quality_edge_demo/`) and
downloads them. On your machine, put the `.pth` in the project's `models/` folder.

In [ ]:
import os, shutil
from google.colab import drive, files

drive.mount("/content/drive")
target = "/content/drive/MyDrive/cv_quality_edge_demo"
os.makedirs(target, exist_ok=True)
for path in (f"models/pcb_{MODEL}.pth", f"models/pcb_{MODEL}.metrics.json"):
    shutil.copy(path, target)
    files.download(path)
print("Saved to", target)